# 01 — Exploratory data analysis

IBM Telco Customer Churn: distributions, **key categorical drivers** (contract, internet, payment), numeric correlations, and data-quality notes that inform the training pipeline.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent))
from src.data.load_data import load_raw_data

try:
    plt.style.use("ggplot")
except Exception:
    pass
%matplotlib inline

In [ ]:
RAW = Path("../data/raw/telco_customer_churn.csv")
df = load_raw_data(RAW)
print("Shape:", df.shape)
display(df.head())
display(df.describe(include="all").T.head(15))

## Data types and missing values

In [ ]:
display(df.dtypes)
print("Null counts (top):")
print(df.isnull().sum().sort_values(ascending=False).head(10))

## Target: churn rate (~26–27% in this dataset)

Imbalanced → `class_weight='balanced'` for linear / forest models in training.

In [ ]:
vc = df["Churn"].value_counts(normalize=True)
print(vc.round(3))
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
df["Churn"].value_counts().plot(kind="bar", ax=ax[0], color=["#2ecc71", "#e74c3c"], edgecolor="black")
ax[0].set_title("Churn counts")
ax[0].set_xticklabels(ax[0].get_xticklabels(), rotation=0)
ax[1].pie(vc, labels=vc.index, autopct="%1.1f%%", colors=["#2ecc71", "#e74c3c"])
ax[1].set_title("Churn share")
plt.tight_layout()
plt.show()

## Tenure

Short tenure often correlates with higher churn (explored with contract below).

In [ ]:
display(df["tenure"].describe())
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df["tenure"], bins=36, edgecolor="black", alpha=0.75, color="steelblue")
ax.set_xlabel("Tenure (months)")
ax.set_ylabel("Customers")
ax.set_title("Tenure distribution")
plt.tight_layout()
plt.show()

## Contract type vs churn

**Month-to-month** customers churn at much higher rates — strong business signal.

In [ ]:
rate = df.groupby("Contract")["Churn"].apply(lambda s: (s == "Yes").mean()).sort_values(ascending=False)
display(pd.DataFrame({"churn_rate": rate.round(3)}))
ct = pd.crosstab(df["Contract"], df["Churn"])
ct = ct.reindex(["Month-to-month", "One year", "Two year"], fill_value=0)
fig, ax = plt.subplots(figsize=(8, 4))
ct.plot(kind="bar", ax=ax, color=["#2ecc71", "#e74c3c"], edgecolor="black")
ax.set_title("Churn by contract type")
ax.set_xlabel("Contract")
ax.legend(title="Churn")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## Internet service & add-ons

Fiber + lack of security/backup often associates with churn in this dataset.

In [ ]:
for col in ["InternetService", "OnlineSecurity", "TechSupport"]:
    r = df.groupby(col)["Churn"].apply(lambda s: (s == "Yes").mean()).sort_values(ascending=False)
    print(f"\n{col} — churn rate:")
    display(r.round(3))

## Payment method

**Electronic check** often shows the highest churn among payment methods — useful for feature importance narratives.

In [ ]:
pm = df.groupby("PaymentMethod")["Churn"].apply(lambda s: (s == "Yes").mean()).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(9, 4))
pm.plot(kind="barh", ax=ax, color="coral", edgecolor="black")
ax.set_xlabel("Churn rate")
ax.set_title("Churn rate by payment method")
plt.tight_layout()
plt.show()

## Other binary / low-cardinality signals

In [ ]:
for col in ["PaperlessBilling", "Partner", "Dependents", "SeniorCitizen"]:
    if col in df.columns:
        r = df.groupby(col)["Churn"].apply(lambda s: (s == "Yes").mean())
        print(col, ":", r.round(3).to_dict())

## Numeric features vs churn

Point-biserial style correlation with binary churn (0/1).

In [ ]:
d = df.copy()
d["churn_01"] = (d["Churn"] == "Yes").astype(int)
d["TotalCharges_num"] = pd.to_numeric(d["TotalCharges"], errors="coerce")
num = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges_num", "churn_01"]
cm = d[num].corr()
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(num)))
ax.set_yticks(range(len(num)))
ax.set_xticklabels(num, rotation=45, ha="right")
ax.set_yticklabels(num)
for i in range(len(num)):
    for j in range(len(num)):
        ax.text(j, i, f"{cm.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.colorbar(im, ax=ax)
ax.set_title("Correlation matrix (numeric + churn)")
plt.tight_layout()
plt.show()
cc = cm["churn_01"].drop("churn_01")
print("Correlation with churn (by |corr|):")
print(cc.reindex(cc.abs().sort_values(ascending=False).index).round(3))

## MonthlyCharges and TotalCharges by churn

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
df.boxplot(column="MonthlyCharges", by="Churn", ax=axes[0])
axes[0].set_title("MonthlyCharges")
axes[0].set_xlabel("Churn")
df["TotalCharges_num"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
tmp = df.dropna(subset=["TotalCharges_num"])
tmp.boxplot(column="TotalCharges_num", by="Churn", ax=axes[1])
axes[1].set_title("TotalCharges (coerced)")
axes[1].set_xlabel("Churn")
plt.suptitle("")
plt.tight_layout()
plt.show()

## TotalCharges data quality

In [ ]:
tc = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("Rows with missing/invalid TotalCharges:", tc.isna().sum())
display(tc.dropna().describe())

## Takeaways for modeling

1. **Class imbalance** → balanced class weights for linear / RF; GB has no `class_weight` in sklearn (handled in design doc).
2. **Contract** and **tenure** are strong segments — `tenure_group` buckets help tree models.
3. **PaymentMethod** (electronic check) and **fiber** internet patterns align with churn.
4. **TotalCharges** must be coerced + imputed (median in pipeline).
5. Next: engineered features in `02_feature_engineering.ipynb`.